In [2]:
import sys, pandas as pd
print(sys.executable)
print(pd.__version__)

d:\Personal Work\Resume-Projects\ppe-compliance-detector\.venv\Scripts\python.exe
3.0.6


In [3]:
import json

with open("../data/CUAD_v1/CUAD_v1.json", encoding="utf-8") as f:
    cuad = json.load(f)

contracts = cuad["data"]
print(len(contracts))
print(contracts[0]["title"])

510
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT


In [4]:
c = contracts[0]
text = c["paragraphs"][0]["context"]
print(c["title"])
print("characters:", len(text))
print(text[:1500])

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
characters: 54290
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  the 

In [5]:
qas = c["paragraphs"][0]["qas"]
print("questions for this contract:", len(qas))
print(qas[0])

questions for this contract: 41
{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}


In [6]:
for qa in qas:
    if qa["answers"]:
        print(qa["id"].split("__")[-1])
        for a in qa["answers"]:
            print("  start:", a["answer_start"], "|", a["text"][:200])
        print()

Document Name
  start: 44 | DISTRIBUTOR AGREEMENT

Parties
  start: 244 | Distributor
  start: 148 | Electric City Corp.
  start: 49574 | Electric City of Illinois L.L.C.
  start: 197 | Company
  start: 212 | Electric City of Illinois LLC

Agreement Date
  start: 263 | 7th day of September, 1999.

Effective Date
  start: 5268 | The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distribu
  start: 31058 | Unless  earlier   terminated   otherwise  provided                   therein,  this  Agreement,  subject to the  commencement  date                   established  in Section 1.3,  shall be effective  

Expiration Date
  start: 5268 | The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distribu

Renewal Term
  s

In [7]:
qa = next(q for q in qas if q["answers"])
a = qa["answers"][0]
s = a["answer_start"]
print(text[s : s + len(a["text"])] == a["text"])

True


In [8]:
import pandas as pd

rows = []
for c in contracts:
    for qa in c["paragraphs"][0]["qas"]:
        rows.append({
            "contract": c["title"],
            "clause": qa["id"].split("__")[-1],
            "has_answer": len(qa["answers"]) > 0,
        })
df = pd.DataFrame(rows)

summary = df.groupby("clause").has_answer.agg(["sum", "count"])
summary["pct_present"] = (summary["sum"] / summary["count"] * 100).round(1)
summary.sort_values("pct_present")

,sum,count,pct_present
clause,,,
Source Code Escrow,13,510,2.5
Price Restrictions,15,510,2.9
Unlimited/All-You-Can-Eat-License,17,510,3.3
Affiliate License-Licensor,23,510,4.5
Most Favored Nation,28,510,5.5
Third Party Beneficiary,32,510,6.3
No-Solicit Of Customers,34,510,6.7
Non-Disparagement,38,510,7.5
Joint Ip Ownership,46,510,9.0


In [9]:
c = contracts[0]
text = c["paragraphs"][0]["context"]
qa = next(q for q in c["paragraphs"][0]["qas"] if q["answers"])
a = qa["answers"][0]
s = a["answer_start"]
print(qa["id"].split("__")[-1])
print(text[s : s + len(a["text"])] == a["text"])

Document Name
True


In [1]:
import sys
sys.path.append("..")

from src.chunking import fixed_chunks, overlaps
from src.data import load_cuad

contracts = load_cuad()
c = contracts[0]
chunks = fixed_chunks(c["text"])
print("contract length:", len(c["text"]), "| chunks:", len(chunks))

gold = c["gold"]["Governing Law"]
print("lawyer highlight position:", gold)

hits = [i for i, ch in enumerate(chunks) if overlaps(ch, gold)]
print("chunks that contain the clause:", hits)
print()
print(chunks[hits[0]]["text"])

contract length: 54290 | chunks: 68
lawyer highlight position: [(52061, 52151)]
chunks that contain the clause: [64, 65]

s  (the          "Sample" or  "Samples")  for use by  Distributor  in  promoting  sales.          Distributor shall use the Samples for purposes of permitting  potential          customers  to use the Products in the field.  The Samples  purchased by          Distributor hereunder shall count toward the minimum expectations under          this Agreement.

6.7      Time of the Essence.  Time is of the essence of this Agreement.

                                    Page -13-

6.8      Force Majeure.  Neither party to this Agreement  shall be liable to the          other  party,  nor shall be subject to  injunctive  relief by the other          party if that party's  performance of its duties or  obligations  under          this  Agreement  is the  consequence  of Force  Majeure  as  defined in          Section 2.2 hereunder.

6.9      Governing Law.  This Agreement is